# Colab runner for the GPU experiments

Thin driver around `scripts/run_experiment.py`; all logic lives in the repository.

1. Runtime -> Change runtime type -> **A100 GPU**.
2. Add `HF_TOKEN` in Colab secrets (needed for MedGemma; accept its terms on the model page).
3. Run the cells in order. Runs are resumable: re-running skips finished runs.

Nothing is pushed to the Hugging Face Hub or GitHub. Results are written to `results/` in the
clone and, optionally, mirrored to Google Drive.

In [ ]:
import os, subprocess
REPO = "https://github.com/Henryayay/AI-xray-classification-project.git"
BRANCH = "cursor/rigorous-paper-pipeline-a80b"
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, "/content/repo"], check=True)
%cd /content/repo
!git log -1 --oneline

In [ ]:
%pip install -q -r requirements.txt -r requirements-gpu.txt

In [ ]:
# Optional: keep results on Google Drive so they survive runtime resets.
USE_DRIVE = True
RESULTS = "results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/cxr_results"
    os.makedirs(RESULTS, exist_ok=True)
    # Seed Drive with the committed results (audit, power, legacy) without overwriting runs.
    !rsync -a --ignore-existing results/ {RESULTS}/

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:
    print("HF_TOKEN not set:", exc)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Experiments

`FINAL = False` scores only the validation split (for development). Set `FINAL = True` only
after `docs/analysis_plan.md` is frozen; every final run is appended to
`results/test_access_log.jsonl`.

In [ ]:
FINAL = True
CONFIGS = [
    "cnn_resnet50", "cnn_resnet50_data_efficiency", "cnn_densenet121", "cnn_xrv_densenet121",
    "probe_rad_dino", "probe_imagenet_resnet50",
    "vlm_gemma3_zeroshot", "vlm_medgemma_zeroshot",
    "vlm_gemma3_icl_random", "vlm_medgemma_icl_random",
    "vlm_gemma3_icl_knn", "vlm_medgemma_icl_knn",
    "vlm_gemma3_lora", "vlm_medgemma_lora", "vlm_gemma3_lora_data_efficiency",
]
for name in CONFIGS:
    flag = "--final" if FINAL else ""
    !python scripts/run_experiment.py --config configs/{name}.yaml --results {RESULTS} --cache {RESULTS}/../cxr_cache {flag}

In [ ]:
# Grad-CAM for the CNN baselines (requires final runs).
for task in ("task1", "task2"):
    for exp in ("cnn_resnet50", "cnn_densenet121", "cnn_xrv_densenet121"):
        !python scripts/gradcam.py --task {task} --experiment {exp} --run seed0 --results {RESULTS}

In [ ]:
# Analysis, tables and figures. Copy RESULTS back into the repo's results/ and commit them.
!python scripts/analyze.py --results {RESULTS}
!python scripts/make_tables.py --results {RESULTS}
!python scripts/make_figures.py --results {RESULTS}
!pip freeze > {RESULTS}/environment-gpu.lock